# 01 · Raccolta e qualità dei dati

Scarichiamo le stagioni configurate, controlliamo copertura e valori mancanti, poi prepariamo le probabilità implicite. La logica vive nel package `football_odds`: il notebook serve per esplorare.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

import pandas as pd
from football_odds.config import AnalysisConfig
from football_odds.data import load_all_seasons
from football_odds.odds import find_odds_columns, prepare_matches

pd.set_option("display.max_columns", 30)

In [ ]:
config = AnalysisConfig(project_dir=PROJECT_DIR)
raw = load_all_seasons(config)
raw.shape

In [ ]:
quality = (
    raw.groupby("Season")
    .agg(partite=("FTR", "size"), risultati_presenti=("FTR", "count"))
)
quality

In [ ]:
odds_columns = find_odds_columns(raw)
print("Quote selezionate:", odds_columns)
raw[list(odds_columns)].isna().groupby(raw["Season"]).mean().style.format("{:.1%}")

In [ ]:
matches = prepare_matches(raw)
matches[["Season", "Date", "HomeTeam", "AwayTeam", "FTR", "p_home", "p_draw", "p_away", "margin"]].head()

In [ ]:
matches.groupby("Season").agg(
    partite=("FTR", "size"),
    margine_medio=("margin", "mean"),
    probabilita_totale=("overround", "mean"),
).style.format({"margine_medio": "{:.2%}", "probabilita_totale": "{:.3f}"})